# GMM-VAE open-set trên MNIST

**Ý tưởng thí nghiệm** (thay cho scRNA-seq → ST):

| Vai trò trong notebook gốc | Trên MNIST |
|---|---|
| scRNA-seq (tập tham chiếu, có nhãn) | Ảnh MNIST **train** của các chữ số *đã biết* — chữ số bị giấu bị **xoá khỏi** tập này |
| ST (tập không nhãn) | Ảnh MNIST **test** của **cả 10 chữ số** — nhãn không dùng khi huấn luyện |
| Unknown / OOD | Chữ số bị giấu (`unknown_classes`): chỉ có ở tập không nhãn |
| 2000 HVG | 784 điểm ảnh (28×28), chuẩn hoá về [0, 1] |

Tập tham chiếu và tập không nhãn được đưa **cùng nhau** qua encoder để học một không gian ẩn chung, sau đó dùng đúng pipeline gán nhãn open-set `hybrid_thr | margin=per_class | ent=OFF | sep=OFF`.

> Chạy trên Colab: **Runtime → Change runtime type → GPU**.

# 1. Cài đặt & cấu hình

In [ ]:
import os

REPO_URL = "https://github.com/doanhailong8197-ai/spatial-gmm-vae.git"
REPO_NAME = "spatial-gmm-vae"

if not os.path.isdir(f"/content/{REPO_NAME}"):
    !git clone -q {REPO_URL} /content/{REPO_NAME}
%cd /content/{REPO_NAME}/Methods

!pip install -q scanpy anndata scikit-learn umap-learn

import math
import random
import itertools
import warnings
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from torch.utils.data import Dataset, DataLoader
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from matplotlib.lines import Line2D

# model.py lấy từ repo của bạn (không cần preprocess.py: MNIST có tiền xử lý riêng ở dưới)
from model import Encoder, Decoder, GMM_VAE, pretrain_vae, initialize_gmm_parameters, train_phased_gmm_vae

warnings.filterwarnings("ignore")
sns.set_theme(style="white", context="notebook", font_scale=1.0)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 100,
                     "axes.spines.top": False, "axes.spines.right": False})

CFG = dict(
    # ---------------- Thiết lập bài toán open-set ----------------
    unknown_classes=[9],      # chữ số bị GIẤU (xoá khỏi tập tham chiếu). Có thể đặt nhiều lớp, vd [8, 9]
    n_ref_per_class=3000,     # tối đa số ảnh / lớp trong tập tham chiếu (lấy từ MNIST train)
    n_target=None,            # None = dùng toàn bộ MNIST test (10000 ảnh) làm tập không nhãn
    target_shift=None,        # None = không dịch miền. Ví dụ dict(rotate_deg=15, noise_std=0.10) để mô phỏng domain shift
    # ---------------- Mạng ----------------
    batch_size=256,
    latent_dim=16,
    hidden_dims=(512, 256, 128),   # encoder 784-512-256-128-16 ; decoder đối xứng
    max_cells_plot=8000,
    seed=42,
    # ---------------- Pretrain ----------------
    lr_pretrain=1e-3,
    epochs_pretrain=30,
    # ---------------- Huấn luyện chung (3 pha A/B/C) ----------------
    lr_joint=5e-4,
    lambda_1=1.0,
    lambda_2=5.0,
    epochs_A=(3, 5), acc_stop_A=0.90,
    epochs_B=(8, 12), ratio_stop_B=1.80,
    epochs_C=20,
    pull_quantile=0.25,
    repel_weight=2.0,
    radial_weight_B=1.0, radial_weight_C=1.0,
    ood_prior=None,           # None = tự đặt bằng tỉ lệ chữ số bị giấu trong tập không nhãn
)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# 2. Đọc dữ liệu MNIST & tiền xử lý

In [ ]:
def load_mnist():
    """Trả về (X_train uint8 [N,28,28], y_train, X_test, y_test)."""
    try:
        from torchvision import datasets
        tr = datasets.MNIST("/content/mnist", train=True, download=True)
        te = datasets.MNIST("/content/mnist", train=False, download=True)
        return tr.data.numpy(), tr.targets.numpy(), te.data.numpy(), te.targets.numpy()
    except Exception as e:                      # dự phòng nếu mirror của torchvision lỗi
        print("torchvision lỗi -> dùng OpenML:", repr(e)[:120])
        from sklearn.datasets import fetch_openml
        X, y = fetch_openml("mnist_784", version=1, as_frame=False, return_X_y=True)
        X = X.reshape(-1, 28, 28).astype(np.uint8); y = y.astype(int)
        return X[:60000], y[:60000], X[60000:], y[60000:]


def to_unit_float(imgs_uint8):
    """(N,28,28) uint8 -> (N,784) float32 trong [0,1]. Khớp với Sigmoid ở decoder."""
    return imgs_uint8.reshape(len(imgs_uint8), -1).astype(np.float32) / 255.0


def apply_target_shift(X, rotate_deg=0.0, noise_std=0.0, seed=0):
    """Tuỳ chọn: tạo domain shift cho tập không nhãn (xoay ngẫu nhiên + nhiễu Gauss)."""
    g = torch.Generator().manual_seed(seed)
    x = torch.from_numpy(X).view(-1, 1, 28, 28)
    if rotate_deg > 0:
        ang = (torch.rand(len(x), generator=g) * 2 - 1) * math.radians(rotate_deg)
        theta = torch.zeros(len(x), 2, 3)
        theta[:, 0, 0], theta[:, 0, 1] = torch.cos(ang), -torch.sin(ang)
        theta[:, 1, 0], theta[:, 1, 1] = torch.sin(ang), torch.cos(ang)
        grid = F.affine_grid(theta, list(x.shape), align_corners=False)
        x = F.grid_sample(x, grid, align_corners=False, padding_mode="zeros")
    if noise_std > 0:
        x = x + noise_std * torch.randn(x.shape, generator=g)
    return x.clamp(0, 1).reshape(len(x), -1).numpy()


class DigitDataset(Dataset):
    """Cùng giao diện với STDataset của repo: .data, .labels và __getitem__ -> (x, y, idx)."""
    def __init__(self, X, y):
        self.data = torch.from_numpy(np.ascontiguousarray(X, dtype=np.float32))
        self.labels = np.asarray(y, dtype=np.int64)

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        return self.data[idx], torch.tensor(self.labels[idx], dtype=torch.long), idx


# ---------------- Đọc & chia dữ liệu ----------------
Xtr_raw, ytr_digit, Xte_raw, yte_digit = load_mnist()
unknown_digits = sorted(CFG["unknown_classes"])
known_digits = [d for d in range(10) if d not in unknown_digits]
rng_data = np.random.default_rng(CFG["seed"])

# --- Tập THAM CHIẾU (có nhãn): chỉ các chữ số đã biết, lớp bị giấu bị xoá hoàn toàn ---
ref_idx = []
for d in known_digits:
    ids = np.where(ytr_digit == d)[0]
    if len(ids) > CFG["n_ref_per_class"]:
        ids = rng_data.choice(ids, CFG["n_ref_per_class"], replace=False)
    ref_idx.append(ids)
ref_idx = np.sort(np.concatenate(ref_idx))
X_ref = to_unit_float(Xtr_raw[ref_idx])
ref_digit = ytr_digit[ref_idx]

# --- Tập KHÔNG NHÃN: MNIST test, GỒM CẢ chữ số bị giấu ---
tgt_idx = np.arange(len(Xte_raw))
if CFG["n_target"] is not None and CFG["n_target"] < len(tgt_idx):
    tgt_idx = np.sort(rng_data.choice(tgt_idx, CFG["n_target"], replace=False))
X_tgt = to_unit_float(Xte_raw[tgt_idx])
tgt_digit = yte_digit[tgt_idx]
if CFG["target_shift"]:
    X_tgt = apply_target_shift(X_tgt, seed=CFG["seed"], **CFG["target_shift"])
    print("Đã áp dụng domain shift cho tập không nhãn:", CFG["target_shift"])

# --- Ánh xạ nhãn: chỉ gồm các lớp ĐÃ BIẾT (giống label_mapping của tập tham chiếu) ---
label_mapping = {str(d): i for i, d in enumerate(known_digits)}
idx_to_label = {v: k for k, v in label_mapping.items()}
class_names = [idx_to_label[i] for i in range(len(idx_to_label))]

y_ref_idx = np.array([label_mapping[str(d)] for d in ref_digit], dtype=np.int64)
y_tgt_idx = np.array([label_mapping.get(str(d), -1) for d in tgt_digit], dtype=np.int64)   # -1 = Unknown

# Nhãn gốc dạng chuỗi (dùng để vẽ Ground Truth và gom "Unknown")
labels_rna_original = np.array([str(d) for d in ref_digit])
labels_spa_original = np.array([str(d) for d in tgt_digit])

dl_rna_ds = DigitDataset(X_ref, y_ref_idx)
dl_spa_ds = DigitDataset(X_tgt, y_tgt_idx)
dl_rna = DataLoader(dl_rna_ds, batch_size=CFG["batch_size"], shuffle=True, drop_last=True)
dl_spa = DataLoader(dl_spa_ds, batch_size=CFG["batch_size"], shuffle=True, drop_last=True)
ds_rna, ds_spa = dl_rna.dataset, dl_spa.dataset

input_dim = ds_rna.data.shape[1]          # 784
n_clusters = len(label_mapping)           # số lớp đã biết

# ood_prior của kênh dustbin: mặc định = tỉ lệ chữ số bị giấu trong tập không nhãn
frac_unknown = float((y_tgt_idx < 0).mean())
if CFG["ood_prior"] is None:
    CFG["ood_prior"] = float(np.clip(round(frac_unknown, 2), 0.05, 0.5))

print(f"Tham chiếu: {len(ds_rna)} ảnh | Không nhãn: {len(ds_spa)} ảnh | pixel: {input_dim} | lớp đã biết: {n_clusters}")
print(f"Chữ số bị giấu: {unknown_digits} -> {frac_unknown:.1%} tập không nhãn | ood_prior = {CFG['ood_prior']}")

cnt = pd.DataFrame({
    "Tham chiếu (có nhãn)": pd.Series(ref_digit).value_counts(),
    "Không nhãn (nhãn ẩn)": pd.Series(tgt_digit).value_counts(),
}).reindex(range(10)).fillna(0).astype(int)
cnt["Vai trò"] = ["Unknown (bị giấu)" if d in unknown_digits else "Known" for d in range(10)]
display(cnt)

fig, axes = plt.subplots(2, 10, figsize=(14, 3.4))
for d in range(10):
    a = np.where(tgt_digit == d)[0]
    axes[0, d].imshow(X_tgt[a[0]].reshape(28, 28), cmap="gray")
    axes[0, d].set_title(f"{d}" + (" (Unknown)" if d in unknown_digits else ""), fontsize=8,
                         color="red" if d in unknown_digits else "black")
    r = np.where(ref_digit == d)[0]
    if len(r):
        axes[1, d].imshow(X_ref[r[0]].reshape(28, 28), cmap="gray")
    else:
        axes[1, d].text(0.5, 0.5, "không có\ntrong\ntham chiếu", ha="center", va="center", fontsize=7, color="red")
    for a_ in (axes[0, d], axes[1, d]):
        a_.set_xticks([]); a_.set_yticks([])
axes[0, 0].set_ylabel("Không nhãn", fontsize=9); axes[1, 0].set_ylabel("Tham chiếu", fontsize=9)
plt.tight_layout(); plt.show()

# 3. Khởi tạo mô hình

In [ ]:
latent_dim = CFG["latent_dim"]
hidden_dims = tuple(CFG["hidden_dims"])      # 784 -> 512 -> 256 -> 128 -> latent 16

encoder = Encoder(input_dim, hidden_dims, latent_dim)
decoder_src = Decoder.from_encoder_dims(hidden_dims, latent_dim, input_dim)
decoder_tgt = Decoder.from_encoder_dims(hidden_dims, latent_dim, input_dim)

gmm_vae = GMM_VAE(encoder, decoder_src, decoder_tgt,
                  n_clusters=n_clusters, latent_dim=latent_dim,
                  input_dim=input_dim).to(device)

optimizer_pre = torch.optim.Adam(gmm_vae.parameters(), lr=CFG["lr_pretrain"])

print(f"hidden_dims: {hidden_dims} | latent_dim: {latent_dim} | số cụm GMM: {n_clusters}")
print(f"Tổng tham số: {sum(p.numel() for p in gmm_vae.parameters()):,}")

# 4. Pretrain VAE (reconstruction trên cả tập tham chiếu và tập không nhãn)

In [ ]:
CKPT_PATH = "/content/gmm_vae_pretrain_mnist.pt"
RESUME_PRETRAIN = False        # True: nạp checkpoint đã lưu, bỏ qua bước pretrain

if RESUME_PRETRAIN and os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
    gmm_vae.load_state_dict(ckpt["model_state"])
    optimizer_pre.load_state_dict(ckpt["optimizer_state"])
    hist_pre = ckpt["hist_pre"]
    gmm_vae.to(device).eval()
    print("Đã nạp checkpoint pretrain, bỏ qua huấn luyện.")
else:
    hist_pre = pretrain_vae(gmm_vae, dl_rna, optimizer_pre, CFG["epochs_pretrain"],
                            device, dataloader_spa=dl_spa)
    torch.save({"model_state": gmm_vae.state_dict(), "optimizer_state": optimizer_pre.state_dict(),
                "hist_pre": hist_pre, "epochs_pretrain": CFG["epochs_pretrain"], "cfg": CFG}, CKPT_PATH)
    print(f"Đã lưu checkpoint: {CKPT_PATH}")

plt.figure(figsize=(5, 3.2))
plt.plot(range(1, len(hist_pre) + 1), hist_pre, color="#3b6fb6", lw=2)
plt.xlabel("Epoch"); plt.ylabel("Reconstruction loss"); plt.title("Pre-training")
plt.tight_layout(); plt.show()

## K-Means/GMM: khởi tạo pi, mu_c, var_c từ tập tham chiếu có nhãn

In [ ]:
info = initialize_gmm_parameters(gmm_vae, dl_rna, n_clusters, device, seed=CFG["seed"])
label_to_cluster = info["label_to_cluster"]

assert (label_to_cluster >= 0).all(), "Có nhãn chưa được ghép cụm: kiểm tra n_clusters"
print("Ánh xạ nhãn -> cụm GMM:")
for lbl, clu in enumerate(label_to_cluster.tolist()):
    print(f"  chữ số {class_names[lbl]:<3} -> cụm {clu}")
print("pi:", np.round(info["pi"].cpu().numpy(), 3))

# 5. Huấn luyện chung (3 pha: A → B → C)

In [ ]:
optimizer_joint = torch.optim.Adam(gmm_vae.parameters(), lr=CFG["lr_joint"])

# Tổng phương sai của dữ liệu, chỉ dùng để báo cáo R² (thay cho hằng số 17.204 / 22.754 của dữ liệu sinh học)
total_var_rna = float(ds_rna.data.var(0).sum())
total_var_spa = float(ds_spa.data.var(0).sum())
print(f"total_var: rna={total_var_rna:.2f} | spa={total_var_spa:.2f}")

hist = train_phased_gmm_vae(
    gmm_vae, dl_rna, dl_spa, optimizer_joint,
    lambda_1=CFG["lambda_1"],
    lambda_2=CFG["lambda_2"],
    device=device,
    label_to_cluster=label_to_cluster,
    epochs_A=CFG["epochs_A"], acc_stop_A=CFG["acc_stop_A"],
    epochs_B=CFG["epochs_B"], ratio_stop_B=CFG["ratio_stop_B"],
    epochs_C=CFG["epochs_C"],
    pull_quantile=CFG["pull_quantile"],
    ood_prior=CFG["ood_prior"],
    repel_weight=CFG["repel_weight"],
    radial_weight_B=CFG["radial_weight_B"],
    radial_weight_C=CFG["radial_weight_C"],
    total_var_rna=total_var_rna, total_var_spa=total_var_spa,
)

In [ ]:
hist_df = pd.DataFrame(hist)
hist_df.index += 1
hist_df.index.name = "epoch"

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, cols in zip(axes, [["total"], ["recon", "gmm"], ["align", "var", "pull"]]):
    for c in cols:
        ax.plot(hist_df.index, hist_df[c], lw=2, label=c)
    ax.set_xlabel("Epoch"); ax.legend()
axes[0].set_title("Total"); axes[1].set_title("Recon / GMM"); axes[2].set_title("Align (ot) / var / pull")
plt.tight_layout(); plt.show()

# 6. Trích xuất không gian ẩn

In [ ]:
@torch.no_grad()
def extract_latent(model, dataset, batch_size, device):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, drop_last=False)
    zs, ys, gs = [], [], []
    for x, y, _ in loader:
        mu, _ = model.encoder(x.to(device))
        zs.append(mu.cpu())
        gs.append(model.get_gamma(mu).cpu())
        ys.append(y)
    return torch.cat(zs).numpy(), torch.cat(ys).numpy(), torch.cat(gs).numpy()

gmm_vae.eval()
Z_rna, y_rna, gamma_rna = extract_latent(gmm_vae, ds_rna, CFG["batch_size"], device)
Z_spa, y_spa, gamma_spa = extract_latent(gmm_vae, ds_spa, CFG["batch_size"], device)
print("Z_rna:", Z_rna.shape, "| Z_spa:", Z_spa.shape)

# 7. Trực quan hoá không gian ẩn

In [ ]:
def run_umap(Z, seed):
    ad_ = sc.AnnData(np.asarray(Z, dtype=np.float32))
    sc.pp.neighbors(ad_, n_neighbors=15, use_rep="X")
    sc.tl.umap(ad_, random_state=seed)
    return ad_.obsm["X_umap"].copy()

rng = np.random.default_rng(CFG["seed"])
n_plot = min(CFG["max_cells_plot"], len(Z_rna), len(Z_spa))
idx_r = np.sort(rng.choice(len(Z_rna), n_plot, replace=False))
idx_s = np.sort(rng.choice(len(Z_spa), n_plot, replace=False))

Z_plot = np.vstack([Z_rna[idx_r], Z_spa[idx_s]])
domain = np.array(["Reference (có nhãn)"] * n_plot + ["Target (không nhãn)"] * n_plot)
is_rna = domain == "Reference (có nhãn)"

emb_umap = run_umap(Z_plot, CFG["seed"])
emb_tsne = TSNE(n_components=2, perplexity=30, init="pca",
                random_state=CFG["seed"], n_jobs=-1).fit_transform(Z_plot)

embeddings = {"UMAP": emb_umap, "t-SNE": emb_tsne}

In [ ]:
def style_axis(ax, title, name):
    ax.set_title(title, fontsize=12, weight="bold")
    ax.set_xlabel(f"{name} 1"); ax.set_ylabel(f"{name} 2")
    ax.set_xticks([]); ax.set_yticks([])

def scatter_cat(ax, emb, labels, palette, s=5, alpha=0.8):
    for lab, color in palette.items():
        m = labels == lab
        ax.scatter(emb[m, 0], emb[m, 1], s=s, c=[color], alpha=alpha,
                   linewidths=0, rasterized=True, label=lab)

domain_palette = {"Reference (có nhãn)": "#3b6fb6", "Target (không nhãn)": "#e07b39"}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
order = rng.permutation(len(domain))
for ax, (name, emb) in zip(axes, embeddings.items()):
    ax.scatter(emb[order, 0], emb[order, 1], s=5, linewidths=0, rasterized=True, alpha=0.7,
               c=[domain_palette[d] for d in domain[order]])
    style_axis(ax, f"Domain alignment ({name})", name)

handles = [plt.Line2D([], [], marker="o", ls="", color=c, label=k, markersize=7)
           for k, c in domain_palette.items()]
fig.legend(handles=handles, loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout(); plt.savefig("fig1_domain_alignment.png", bbox_inches="tight"); plt.show()

In [ ]:
labels_rna_plot = np.array([class_names[i] for i in y_rna[idx_r]])

type_palette = dict(zip(class_names, sns.color_palette("tab10", len(class_names))))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
for ax, (name, emb) in zip(axes, embeddings.items()):
    scatter_cat(ax, emb[is_rna], labels_rna_plot, type_palette)
    style_axis(ax, f"Tập tham chiếu theo Ground Truth ({name})", name)
axes[1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False,
               markerscale=3, fontsize=9, title="Chữ số")
plt.tight_layout(); plt.savefig("fig2_source_celltype.png", bbox_inches="tight"); plt.show()

In [ ]:
g = gamma_spa[idx_s]
comp = g.argmax(1)
entropy = -(g * np.log(g + 1e-10)).sum(1)
cluster_to_label = {int(c): class_names[l] for l, c in enumerate(label_to_cluster.tolist())}

fig, axes = plt.subplots(2, 2, figsize=(15, 10.5))

for idx, (name, emb_full) in enumerate(embeddings.items()):
    ax = axes[0, idx]
    emb = emb_full[~is_rna]
    for c in np.unique(comp):
        m = comp == c
        cell_type = cluster_to_label.get(c, "?")
        color = type_palette.get(cell_type, "#333333")
        ax.scatter(emb[m, 0], emb[m, 1], s=5, c=[color], linewidths=0, alpha=0.8,
                   rasterized=True, label=f"{c}: {cell_type}" if idx == 1 else "")
    style_axis(ax, f"Tập không nhãn theo thành phần GMM ({name})", name)

axes[0, 1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False,
                  markerscale=3, fontsize=9, title="Cụm: chữ số tương ứng")

for idx, (name, emb_full) in enumerate(embeddings.items()):
    ax = axes[1, idx]
    emb = emb_full[~is_rna]
    sc_ = ax.scatter(emb[:, 0], emb[:, 1], s=5, c=entropy, cmap="magma",
                     linewidths=0, alpha=0.85, rasterized=True)
    style_axis(ax, f"Tập không nhãn theo Entropy của γ ({name})", name)
    cb = plt.colorbar(sc_, ax=ax, fraction=0.046, pad=0.02)
    cb.set_label("Entropy")

plt.tight_layout(); plt.savefig("fig3_target_structure.png", bbox_inches="tight"); plt.show()

In [ ]:
labels_spa_plot = labels_spa_original[idx_s]
new_labels = sorted(set(labels_spa_plot) - set(type_palette.keys()))
if new_labels:
    extra_colors = sns.color_palette("dark", len(new_labels))
    for label, color in zip(new_labels, extra_colors):
        type_palette[label] = color

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
for ax, (name, emb) in zip(axes, embeddings.items()):
    scatter_cat(ax, emb[~is_rna], labels_spa_plot, type_palette)
    style_axis(ax, f"Tập không nhãn theo Ground Truth ({name})", name)
axes[1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False,
               markerscale=3, fontsize=9, title="Ground Truth (gồm chữ số bị giấu)")
plt.tight_layout(); plt.savefig("fig4_target_groundtruth.png", bbox_inches="tight"); plt.show()

In [ ]:
labels_spa_grouped = np.array([l if l in set(class_names) else "Unknown" for l in labels_spa_plot])

palette_unknown = {label: type_palette[label] for label in class_names}
palette_unknown["Unknown"] = "#555555"

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
for ax, (name, emb) in zip(axes, embeddings.items()):
    scatter_cat(ax, emb[~is_rna], labels_spa_grouped, palette_unknown)
    style_axis(ax, f"Tập không nhãn: Ground Truth (OOD masked) - {name}", name)
axes[1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False,
               markerscale=3, fontsize=9, title="Known vs Unknown")
plt.tight_layout(); plt.savefig("fig5_target_groundtruth_unknown.png", bbox_inches="tight"); plt.show()

## Chất lượng phân cụm & mức độ trộn miền

In [ ]:
km_pred = KMeans(n_clusters=n_clusters, n_init=10, random_state=CFG["seed"]).fit_predict(Z_rna)
gmm_pred = gamma_rna.argmax(1)

rows = []
for name, pred in [("K-Means trên Z", km_pred), ("GMM (argmax γ)", gmm_pred)]:
    rows.append({"Phương pháp": name,
                 "ARI": adjusted_rand_score(y_rna, pred),
                 "NMI": normalized_mutual_info_score(y_rna, pred),
                 "Số cụm dùng": len(np.unique(pred))})

metrics_df = pd.DataFrame(rows).set_index("Phương pháp").round(4)
display(metrics_df)

ax = metrics_df[["ARI", "NMI"]].plot.bar(figsize=(5.5, 3.5), color=["#3b6fb6", "#e07b39"], rot=0, width=0.7)
ax.set_ylim(0, 1); ax.set_ylabel("Score"); ax.set_title("Chất lượng phân cụm không gian ẩn (tập tham chiếu)")
for c in ax.containers:
    ax.bar_label(c, fmt="%.3f", fontsize=9, padding=2)
plt.tight_layout(); plt.show()

nn_ = NearestNeighbors(n_neighbors=31).fit(Z_plot)
_, nbr_feat = nn_.kneighbors(Z_plot)
nbr_feat = nbr_feat[:, 1:]
other_frac = (is_rna[nbr_feat] != is_rna[:, None]).mean(1)
print(f"Tỉ lệ láng giềng khác miền (trung bình): {other_frac.mean():.3f}  (lý tưởng ≈ 0.5)")
print(f"  - tham chiếu: {other_frac[is_rna].mean():.3f} | không nhãn: {other_frac[~is_rna].mean():.3f}")

# 8. Gán nhãn open-set (Known vs Unknown)

Các hàm dưới đây giữ nguyên logic của notebook gốc (hình học Mahalanobis + Cosine, chia ngưỡng 95% theo lớp nguồn, margin theo lớp). Chỉ `y_spa` giờ có `-1` cho chữ số bị giấu.

In [ ]:
# ---- Hàm đánh giá open-set (gom từ các cell gốc) ----
def to_idx(cluster, accept):
    return np.where(accept, cluster, -1)

def open_set_metrics(pred, y_true):
    kn, un = y_true >= 0, y_true < 0
    kacc = (pred[kn] == y_true[kn]).mean() if kn.any() else np.nan
    uacc = (pred[un] == -1).mean() if un.any() else np.nan
    tacc = (pred == y_true).mean()
    h = (2.0 * kacc * uacc) / (kacc + uacc + 1e-12)
    return {"accepted": (pred >= 0).mean(), "KACC": kacc, "UACC": uacc, "TACC": tacc, "H-score": h}

In [ ]:
# ==============================================================================
# TÁCH "GÁN NHÃN" (khoảng cách THÔ) KHỎI "TỪ CHỐI OOD" (khoảng cách chia ngưỡng)
# ==============================================================================
K = len(class_names)
y_s_arr = np.asarray(y_rna, dtype=int)
y_t_arr = np.asarray(y_spa, dtype=int)
name_to_idx = {n: i for i, n in enumerate(class_names)}


# ---------- 1. Tính hình học MỘT LẦN, dùng lại cho mọi cấu hình ----------
def build_geometry(Z_target, Z_source, y_source, K, var_shrinkage=0.20,
                   geom_pct=95.0, knn_k=15):
    Z_t = Z_target.astype(np.float64)
    Z_s = Z_source.astype(np.float64)
    y_s = np.asarray(y_source, dtype=int)
    D = Z_s.shape[1]

    mu = np.zeros((K, D))
    var_raw = np.zeros((K, D))
    global_var = np.var(Z_s, axis=0) + 1e-6
    for k in range(K):
        m = (y_s == k)
        if m.sum() >= 5:
            Zk = Z_s[m]
            mu[k] = np.median(Zk, axis=0)
            mad = np.median(np.abs(Zk - mu[k]), axis=0) * 1.4826
            var_raw[k] = np.maximum(0.5 * np.var(Zk, axis=0) + 0.5 * mad ** 2, 1e-4)
        else:
            mu[k] = np.median(Z_s, axis=0)
            var_raw[k] = global_var
    var_sup = (1 - var_shrinkage) * var_raw + var_shrinkage * np.median(var_raw, axis=0, keepdims=True)
    inv_var = 1.0 / (var_sup + 1e-8)

    def maha(Z):
        return np.clip((Z ** 2) @ inv_var.T - (2.0 * Z) @ (mu * inv_var).T
                       + ((mu ** 2) * inv_var).sum(axis=1), 0.0, None)

    d2_s_maha, d2_t_maha = maha(Z_s), maha(Z_t)

    # Euclidean thô tới tâm
    d2_t_euc = np.clip((Z_t ** 2).sum(1, keepdims=True) - 2.0 * Z_t @ mu.T + (mu ** 2).sum(1)[None, :], 0.0, None)
    d_t_euc = np.sqrt(d2_t_euc)

    # Cosine thô
    c_s, c_t = np.median(Z_s, axis=0, keepdims=True), np.median(Z_t, axis=0, keepdims=True)
    Zs_c = Z_s - c_s
    Zt_c = Z_t - (0.5 * c_s + 0.5 * c_t)
    mu_c = mu - c_s
    nrm = lambda A: A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-12)
    d_s_cos = np.clip(1.0 - nrm(Zs_c) @ nrm(mu_c).T, 0.0, 2.0)
    d_t_cos = np.clip(1.0 - nrm(Zt_c) @ nrm(mu_c).T, 0.0, 2.0)

    # Ngưỡng phân vị 95% (CHỈ dùng cho bước từ chối)
    maha_thr = np.zeros(K); cos_thr = np.zeros(K)
    fb_m = np.percentile(d2_s_maha.min(1), geom_pct)
    fb_c = np.percentile(d_s_cos.min(1), geom_pct)
    for k in range(K):
        m = (y_s == k)
        if m.sum() >= 5:
            maha_thr[k] = np.percentile(d2_s_maha[m, k], geom_pct)
            cos_thr[k] = np.percentile(d_s_cos[m, k], geom_pct)
        else:
            maha_thr[k], cos_thr[k] = fb_m, fb_c

    # k-NN xác suất trên Z
    knn = KNeighborsClassifier(n_neighbors=knn_k, weights="distance").fit(Z_s, y_s)
    proba = np.zeros((len(Z_t), K))
    proba[:, knn.classes_] = knn.predict_proba(Z_t)

    return dict(
        K=K, y_s=y_s, n_t=len(Z_t),
        r_s_maha=d2_s_maha / (maha_thr[None, :] + 1e-12),
        r_t_maha=d2_t_maha / (maha_thr[None, :] + 1e-12),
        r_s_cos=d_s_cos / (cos_thr[None, :] + 1e-12),
        r_t_cos=d_t_cos / (cos_thr[None, :] + 1e-12),
        d_t_cos=d_t_cos, d_t_euc=d_t_euc, knn_proba=proba,
    )


# ---------- 2. Điểm dùng để GÁN NHÃN (càng nhỏ càng gần) ----------
def label_scores(geo, mode, cosine_blend=0.65):
    if mode == "hybrid_thr":      # phương pháp hiện tại (có chia ngưỡng) - làm mốc
        return (1 - cosine_blend) * geo["r_t_maha"] + cosine_blend * geo["r_t_cos"]
    if mode == "cos_raw":         # cosine thô, KHÔNG chia ngưỡng
        return geo["d_t_cos"]
    if mode == "euclid_raw":      # Euclidean thô
        return geo["d_t_euc"]
    if mode == "cos_euc_raw":     # trộn 2 thước đo thô, chỉ chia cho 1 hằng số TOÀN CỤC
        return (geo["d_t_cos"] / np.median(geo["d_t_cos"])
                + geo["d_t_euc"] / np.median(geo["d_t_euc"]))
    if mode == "knn":
        return 1.0 - geo["knn_proba"]
    raise ValueError(mode)


# ---------- 3. Cascade v2: nhãn từ điểm thô, từ chối bằng r chia ngưỡng ----------
def cascade_v2(geo, label_mode="hybrid_thr", margin_mode="global", use_entropy=True,
               use_sep=True, sep_ratio_thr=0.90, cosine_blend=0.65, reject_blend=0.65,
               temperature=0.15, source_ent_pct=90.0, coupling_strength=0.35,
               use_hubness=True, shrink_n=50):
    K, y_s, n_t = geo["K"], geo["y_s"], geo["n_t"]
    idx = np.arange(n_t)

    # (a) GÁN NHÃN
    S = label_scores(geo, label_mode, cosine_blend)
    pred = S.argmin(axis=1)

    # (b) Điểm dùng để TỪ CHỐI (chia ngưỡng 95% để so sánh công bằng giữa các lớp)
    r_s = (1 - reject_blend) * geo["r_s_maha"] + reject_blend * geo["r_s_cos"]
    r_t = (1 - reject_blend) * geo["r_t_maha"] + reject_blend * geo["r_t_cos"]
    r_assigned = r_t[idx, pred]
    r_s_assigned = r_s[np.arange(len(y_s)), y_s]

    # (c) Tỷ số tách biệt tính trên CHÍNH điểm dùng để gán nhãn
    S_sorted = np.sort(S, axis=1)
    sep_ratio = S_sorted[:, 0] / (S_sorted[:, 1] + 1e-12)
    pass_sep = (sep_ratio <= sep_ratio_thr) if use_sep else np.ones(n_t, dtype=bool)

    def softmax_ent(r):
        lg = -r / temperature
        lg = lg - lg.max(axis=1, keepdims=True)
        g = np.exp(lg) / (np.exp(lg).sum(axis=1, keepdims=True) + 1e-12)
        return -np.sum(g * np.log(g + 1e-12), axis=1)

    ent_t, ent_s = softmax_ent(r_t), softmax_ent(r_s)

    # (d) Margin toàn cục
    reliable = pass_sep & (ent_t <= np.median(ent_t))
    if reliable.sum() < 100:
        reliable = pass_sep
    sq = np.sqrt(r_assigned[reliable])
    iqr = np.percentile(sq, 75) - np.percentile(sq, 25)
    global_margin = float(np.clip((np.median(sq) + 1.15 * iqr / 1.349) ** 2, 1.4, 3.8))

    # (e) Margin theo lớp (co về margin toàn cục khi lớp có ít điểm)
    margins = np.full(K, global_margin)
    if margin_mode == "per_class":
        for k in range(K):
            pool = reliable & (pred == k)
            n_k = int(pool.sum())
            if n_k >= 10:
                s_k = np.sqrt(r_assigned[pool])
                iqr_k = np.percentile(s_k, 75) - np.percentile(s_k, 25)
                m_k = float(np.clip((np.median(s_k) + 1.15 * iqr_k / 1.349) ** 2, 1.4, 3.8))
                w = n_k / (n_k + shrink_n)
                margins[k] = w * m_k + (1 - w) * global_margin

    # (f) Phạt phình cụm (giữ như bản gốc)
    if use_hubness:
        src_ratio = np.bincount(y_s, minlength=K) / float(len(y_s))
        tgt_ratio = np.bincount(pred, minlength=K) / float(n_t)
        for k in range(K):
            if tgt_ratio[k] > 1.6 * src_ratio[k]:
                pen = ((src_ratio[k] + 0.015) / (tgt_ratio[k] + 0.015)) ** 0.45
                margins[k] = np.clip(margins[k] * pen, 1.0, margins[k])

    st_margins = margins[pred]
    accept_stage1 = (r_assigned <= st_margins) & pass_sep

    # (g) Tầng 2 entropy (có thể bỏ)
    if use_entropy:
        fb = np.percentile(ent_s, source_ent_pct)
        class_thr = np.zeros(K)
        for k in range(K):
            m = (y_s == k) & (r_s_assigned <= 1.0)
            class_thr[k] = np.percentile(ent_s[m], source_ent_pct) if m.sum() >= 5 else fb
        src_in = ent_s[r_s_assigned <= 1.0]
        tgt_in = ent_t[accept_stage1] if accept_stage1.sum() > 50 else ent_t
        scale = float(np.clip(np.sqrt(np.maximum(np.median(tgt_in) / (np.median(src_in) + 1e-6), 1.0)), 1.0, 1.30))
        base = np.clip(class_thr[pred] * scale, 0.20, np.log(2.0))
        rel = np.clip(r_assigned / (st_margins + 1e-12), 0.0, 1.0)
        dyn = np.clip(base * (1.0 + coupling_strength * (1.0 - 2.0 * rel)), 0.15, np.log(2.0))
        accept = accept_stage1 & (ent_t <= dyn)
    else:
        accept = accept_stage1

    known = y_t_arr >= 0
    return dict(pred=pred, accept=accept, accept_stage1=accept_stage1,
                raw_kacc=float((pred[known] == y_t_arr[known]).mean()),
                margin_mean=float(st_margins.mean()), margins=margins)


def evaluate(res):
    m = open_set_metrics(to_idx(res["pred"], res["accept"]), y_spa)   # open_set_metrics / to_idx ở cell phía trên
    m["raw_KACC"] = res["raw_kacc"]
    m["margin"] = res["margin_mean"]
    return m


def per_class_acc(pred, accept=None):
    out = {}
    for k, n in enumerate(class_names):
        m = (y_t_arr == k)
        if m.sum() == 0:
            continue
        ok = (pred[m] == k) if accept is None else ((pred[m] == k) & accept[m])
        out[n] = round(float(ok.mean()) * 100, 1)
    return out

In [ ]:
# ---------- So sánh 5 cách gán nhãn (margin toàn cục, entropy bật = pipeline gốc) ----------
geo = build_geometry(Z_spa, Z_rna, y_rna, K)
modes = ["hybrid_thr", "cos_raw", "euclid_raw", "cos_euc_raw", "knn"]

raw_tbl, fin_tbl, summ = {}, {}, {}
for md_ in modes:
    res_ = cascade_v2(geo, label_mode=md_)
    raw_tbl[md_] = per_class_acc(res_["pred"])                    # Raw argmax (chưa lọc OOD)
    fin_tbl[md_] = per_class_acc(res_["pred"], res_["accept"])    # Sau lọc OOD
    summ[md_] = evaluate(res_)

print("=== (A) ĐỘ CHÍNH XÁC RAW ARGMAX THEO LỚP (%) - chưa lọc OOD ===")
display(pd.DataFrame(raw_tbl))
print("\n=== (B) ĐỘ CHÍNH XÁC CUỐI CÙNG THEO LỚP (%) - sau lọc OOD ===")
display(pd.DataFrame(fin_tbl))
print("\n=== (C) CHỈ SỐ OPEN-SET TỔNG HỢP ===")
display(pd.DataFrame(summ).T.round(4))

## Quét cấu hình (ablation) & chốt phương pháp

In [ ]:
FINAL_CFG = dict(label_mode="hybrid_thr", margin_mode="per_class", use_entropy=False, use_sep=False)

def cfg_key(md_, mg, ent, sep):
    return f"{md_} | margin={mg} | ent={'on' if ent else 'OFF'} | sep={'on' if sep else 'OFF'}"

grid = list(itertools.product(modes, ["global", "per_class"], [True, False], [True, False]))
rows, store = [], {}
for md_, mg, ent, sep in grid:
    res_ = cascade_v2(geo, label_mode=md_, margin_mode=mg, use_entropy=ent, use_sep=sep)
    m = evaluate(res_)
    key = cfg_key(md_, mg, ent, sep)
    m["Config"] = key
    rows.append(m); store[key] = (res_, per_class_acc(res_["pred"], res_["accept"]))
df = pd.DataFrame(rows).set_index("Config")

baseline_key = cfg_key("hybrid_thr", "global", True, True)
final_key = cfg_key(FINAL_CFG["label_mode"], FINAL_CFG["margin_mode"], FINAL_CFG["use_entropy"], FINAL_CFG["use_sep"])

print("=== TOP 15 CẤU HÌNH THEO H-SCORE (kèm baseline và cấu hình chốt) ===")
top = df.sort_values("H-score", ascending=False).head(15)
extra = [k for k in (baseline_key, final_key) if k not in top.index]
display(pd.concat([top, df.loc[extra]]).round(4))

print("\n=== TÁC ĐỘNG TỪNG THAY ĐỔI RIÊNG LẺ (so với baseline) ===")
abl = {
    "Baseline": baseline_key,
    "Chỉ margin theo lớp": cfg_key("hybrid_thr", "per_class", True, True),
    "Chỉ bỏ Trạm 2 (entropy)": cfg_key("hybrid_thr", "global", False, True),
    "Chỉ bỏ Trạm 1B (sep)": cfg_key("hybrid_thr", "global", True, False),
    "Margin lớp + bỏ entropy": cfg_key("hybrid_thr", "per_class", False, True),
    "Margin lớp + bỏ entropy + bỏ sep (CHỐT)": final_key,
}
display(df.loc[list(abl.values()), ["accepted", "KACC", "UACC", "TACC", "H-score", "raw_KACC", "margin"]]
        .set_axis(list(abl.keys())).round(4))

best_key = df["H-score"].idxmax()
print(f"\nCấu hình tốt nhất theo H-score trên MNIST: [{best_key}]")
print(f"Cấu hình chốt (chọn trước, không tinh chỉnh trên nhãn test): [{final_key}]")
cmp = pd.DataFrame({"Baseline": store[baseline_key][1], "Chốt": store[final_key][1]})
cmp["Δ"] = cmp["Chốt"] - cmp["Baseline"]
print("\n=== ĐỘ CHÍNH XÁC CUỐI CÙNG THEO LỚP (%) ===")
display(cmp)

# 9. Phân tích lỗi trên toàn bộ tập không nhãn (cấu hình chốt)

UMAP chung vẽ **toàn bộ** ảnh không nhãn và một phần tập tham chiếu, nên nhãn luôn khớp tọa độ.

In [ ]:
POINT_S = 2.0
MAX_REF_UMAP = 10000          # số điểm tham chiếu đưa vào UMAP chung (tập không nhãn dùng toàn bộ)
RECOMPUTE_UMAP = False

# 1. Chạy cấu hình chốt ------------------------------------------------------
res = cascade_v2(geo, **FINAL_CFG)
m_final = evaluate(res)
print("=== METRIC CẤU HÌNH CHỐT ===")
display(pd.Series(m_final).round(4).to_frame("value").T)

K = len(class_names)
y_spa = np.asarray(y_spa).astype(int)
N = len(y_spa)
pred_raw = np.asarray(res["pred"]).astype(int)
acc_spa = np.asarray(res["accept"]).astype(bool) & (pred_raw >= 0)
raw_idx_full = pred_raw
pred_idx_full = np.where(acc_spa, pred_raw, -1)
margins_cls = np.asarray(res["margins"], dtype=float)

def to_names_arr(idx):
    return np.array([class_names[i] if i >= 0 else "Unknown" for i in idx])

raw_plot, pred_plot, true_plot = to_names_arr(raw_idx_full), to_names_arr(pred_idx_full), to_names_arr(y_spa)
is_known, is_ood = y_spa >= 0, y_spa < 0

# 2. UMAP chung: toàn bộ ST + một phần tham chiếu ------------------------------
rng_u = np.random.default_rng(0)
n_ref_u = min(MAX_REF_UMAP, len(Z_rna))
ref_u = np.sort(rng_u.choice(len(Z_rna), n_ref_u, replace=False))
if (not RECOMPUTE_UMAP and "emb_joint_full" in globals() and emb_joint_full.shape[0] == n_ref_u + N):
    print("Dùng lại emb_joint_full đã tính.")
else:
    print(f"Tính UMAP chung cho {n_ref_u} ảnh tham chiếu + {N} ảnh không nhãn ...")
    emb_joint_full = run_umap(np.vstack([Z_rna[ref_u], Z_spa]), CFG["seed"])
emb_rna_full = emb_joint_full[:n_ref_u]
emb_all = emb_joint_full[n_ref_u:]
rna_plot = to_names_arr(y_rna[ref_u])
xlim = (emb_joint_full[:, 0].min() - 0.5, emb_joint_full[:, 0].max() + 0.5)
ylim = (emb_joint_full[:, 1].min() - 0.5, emb_joint_full[:, 1].max() + 0.5)

# Kiểm tra căn chỉnh nhãn-tọa độ
kn_all = np.where(is_known)[0]
kn = rng_u.choice(kn_all, size=min(5000, len(kn_all)), replace=False)
nn = NearestNeighbors(n_neighbors=6).fit(emb_all[kn_all])
_, nb = nn.kneighbors(emb_all[kn])
nb_agree = (y_spa[kn_all][nb[:, 1:]] == y_spa[kn][:, None]).mean()
chance = (pd.Series(y_spa[kn_all]).value_counts(normalize=True) ** 2).sum()
print(f"[Kiểm tra UMAP] láng giềng cùng nhãn thật = {nb_agree:.3f} | ngẫu nhiên ≈ {chance:.3f}")

# 3. Entropy (từ gamma_spa) và Mahalanobis (tính lại từ latent) ----------------
g_ = np.clip(np.asarray(gamma_spa, float), 1e-12, 1.0)
ent_full = -(g_ * np.log(g_)).sum(axis=1)
n_comp = g_.shape[1]

d_dim = Z_spa.shape[1]
d_all = np.full((N, K), np.nan)
for k in range(K):
    Zk = Z_rna[y_rna == k]
    if len(Zk) < 5:
        continue
    mu_k = Zk.mean(0)
    cov = np.cov(Zk.T) + 1e-3 * np.trace(np.cov(Zk.T)) / d_dim * np.eye(d_dim)
    P = np.linalg.pinv(cov)
    diff = Z_spa - mu_k
    d_all[:, k] = np.sqrt(np.einsum("ij,jk,ik->i", diff, P, diff))
d_mah_full = d_all[np.arange(N), np.clip(raw_idx_full, 0, K - 1)]
d_mah_full = np.where(np.isnan(d_mah_full), np.nanmax(d_mah_full), d_mah_full)

# 4. Error taxonomy ---------------------------------------------------------------
TP, WR, FR, FA, TN = ("1. Known: Đúng nhãn (TP)", "2. Known: Gán SAI nhãn (Wrong Class)",
                      "3. Known: Bị lọc nhầm (False Reject)", "4. OOD: Lọt lưới (False Accept)",
                      "5. OOD: Đúng Unknown (TN)")
error_taxonomy = np.empty(N, dtype=object)
error_taxonomy[is_known & acc_spa & (pred_idx_full == y_spa)] = TP
error_taxonomy[is_known & acc_spa & (pred_idx_full != y_spa)] = WR
error_taxonomy[is_known & ~acc_spa] = FR
error_taxonomy[is_ood & acc_spa] = FA
error_taxonomy[is_ood & ~acc_spa] = TN
error_palette = {TP: "#2ca02c", WR: "#d62728", FR: "#ff7f0e", FA: "#9467bd", TN: "#d3d3d3"}

# 5. Ma trận nhầm lẫn ----------------------------------------------------------------
true_cm = np.where(is_known, y_spa, K)
pred_cm = np.where(pred_idx_full >= 0, pred_idx_full, K)
cm = np.zeros((K + 1, K + 1), dtype=int)
np.add.at(cm, (true_cm, pred_cm), 1)
row_sum = cm.sum(axis=1, keepdims=True)
cm_pct = np.divide(cm, row_sum, out=np.zeros(cm.shape, float), where=row_sum > 0) * 100
cm_labels = [f"{c}" for c in class_names] + ["Unknown"]

# 6. Hàm vẽ: Unknown/TN vẽ trước để không che các điểm có màu --------------------
def scatter_ordered(ax, emb, labels, palette, s=POINT_S, alpha=0.8, back=("Unknown", TN), legend=False):
    labels = np.asarray(labels)
    vc = pd.Series(labels).value_counts()
    uniq = [u for u in vc.index if u in back] + [u for u in vc.index[::-1] if u not in back]
    for u in uniq:
        m = labels == u
        ax.scatter(emb[m, 0], emb[m, 1], s=s, alpha=0.35 if u in back else alpha,
                   c=[palette.get(u, "#bbbbbb")], label=f"{u} ({m.sum()})", rasterized=True, linewidths=0)
    if legend:
        ax.legend(loc="best", frameon=True, facecolor="white", framealpha=0.85, markerscale=5, fontsize=8)

def umap_axes(ax, title):
    style_axis(ax, title, "UMAP")
    ax.set_xlim(xlim); ax.set_ylim(ylim)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(26, 23))

scatter_ordered(axes[0, 0], emb_rna_full, rna_plot, type_palette, s=3)
umap_axes(axes[0, 0], f"1. Tham chiếu: Ground Truth (n={n_ref_u}, chỉ chữ số đã biết)")

scatter_ordered(axes[0, 1], emb_all, true_plot, palette_unknown)
umap_axes(axes[0, 1], f"2. Không nhãn: Ground Truth (OOD = Unknown) | N={N}")

raw_kacc = (raw_idx_full[is_known] == y_spa[is_known]).mean()
scatter_ordered(axes[0, 2], emb_all, raw_plot, palette_unknown, back=())
umap_axes(axes[0, 2], f"3. Không nhãn: Nhãn thô hybrid_thr, chưa lọc (raw KACC={raw_kacc:.1%})")

scatter_ordered(axes[1, 0], emb_all, pred_plot, palette_unknown)
umap_axes(axes[1, 0], f"4. Final | acc={acc_spa.mean():.1%} KACC={m_final['KACC']:.3f} "
                      f"UACC={m_final['UACC']:.3f} H={m_final['H-score']:.3f}")

ax_cm = axes[1, 1]
im = ax_cm.imshow(cm_pct, cmap="Blues", vmin=0, vmax=100, aspect="auto")
ax_cm.set_xticks(range(K + 1)); ax_cm.set_yticks(range(K + 1))
ax_cm.set_xticklabels(cm_labels, rotation=60, ha="right", fontsize=9)
ax_cm.set_yticklabels([f"{n} (n={row_sum[i, 0]})" for i, n in enumerate(cm_labels)], fontsize=9)
for i in range(K + 1):
    for j in range(K + 1):
        if cm_pct[i, j] >= 0.5:
            ax_cm.text(j, i, f"{cm_pct[i, j]:.0f}", ha="center", va="center", fontsize=8,
                       color="white" if cm_pct[i, j] > 55 else "black")
ax_cm.axhline(K - 0.5, color="red", lw=1); ax_cm.axvline(K - 0.5, color="red", lw=1)
ax_cm.set_xlabel("Dự đoán"); ax_cm.set_ylabel("Nhãn thật (hàng cuối = chữ số bị giấu)")
ax_cm.set_title("5. Ma trận nhầm lẫn (% theo hàng)", fontsize=10, fontweight="bold")
plt.colorbar(im, ax=ax_cm, fraction=0.046, pad=0.04, label="% theo hàng")

scatter_ordered(axes[1, 2], emb_all, error_taxonomy, error_palette, legend=True)
umap_axes(axes[1, 2], "6. Giải phẫu 5 nhóm lỗi (Error Map)")

ax_b = axes[2, 0]
kept_cls = [acc_spa[y_spa == i].mean() * 100 if (y_spa == i).any() else 0 for i in range(K)]
kacc_cls = [(pred_idx_full[y_spa == i] == i).mean() * 100 if (y_spa == i).any() else 0 for i in range(K)]
xs = np.arange(K); w = 0.38
ax_b.bar(xs - w/2, kept_cls, w, label="Giữ lại sau lọc (%)", color="#1f77b4")
ax_b.bar(xs + w/2, kacc_cls, w, label="KACC cuối (%)", color="#2ca02c")
ax_b.set_xticks(xs)
ax_b.set_xticklabels([f"{n}\nm={margins_cls[i]:.2f}" for i, n in enumerate(class_names)], fontsize=9)
ax_b.set_ylim(0, 105); ax_b.set_ylabel("%")
ax_b.set_title("7. Theo lớp: tỉ lệ giữ / KACC cuối (m = margin lớp)", fontsize=10, fontweight="bold")
ax_b.legend(fontsize=8, loc="lower right")

v_max = np.percentile(d_mah_full, 95.0)
order = np.argsort(d_mah_full)
sc_h = axes[2, 1].scatter(emb_all[order, 0], emb_all[order, 1], c=np.clip(d_mah_full[order], 0, v_max),
                          cmap="viridis", s=POINT_S, alpha=0.8, rasterized=True, linewidths=0)
plt.colorbar(sc_h, ax=axes[2, 1], fraction=0.046, pad=0.04, label="Mahalanobis tới lớp được gán (clip p95)")
umap_axes(axes[2, 1], "8. Bản đồ nhiệt khoảng cách Mahalanobis")

ax_dec = axes[2, 2]
mk_w, mk_r = is_known & (raw_idx_full != y_spa), is_known & (raw_idx_full == y_spa)
ax_dec.scatter(d_mah_full[is_ood], ent_full[is_ood], c="#8c8c8c", s=2.5, alpha=0.35,
               label="True OOD (Unknown)", rasterized=True, linewidths=0)
ax_dec.scatter(d_mah_full[mk_w], ent_full[mk_w], c="#d62728", s=3, alpha=0.45,
               label="Known (nhãn thô sai)", rasterized=True, linewidths=0)
ax_dec.scatter(d_mah_full[mk_r], ent_full[mk_r], c="#2ca02c", s=3, alpha=0.45,
               label="Known (nhãn thô đúng)", rasterized=True, linewidths=0)
ax_dec.set_xlim(0, np.percentile(d_mah_full, 98.0))
ax_dec.set_ylim(-0.02, np.log(n_comp) * 0.75)
ax_dec.set_xlabel("Mahalanobis (d_mah)"); ax_dec.set_ylabel("Shannon entropy (chỉ chẩn đoán)")
ax_dec.set_title("9. Maha vs Entropy (entropy không dùng để lọc)", fontsize=10, fontweight="bold")
ax_dec.legend(loc="upper right", fontsize=8, markerscale=4)

handles = [Line2D([0], [0], marker="o", ls="", ms=8, color=palette_unknown.get(n, "#bbb"), label=n)
           for n in class_names + ["Unknown"]]
fig.legend(handles=handles, loc="lower center", ncol=min(K + 1, 10), frameon=False, fontsize=10,
           title="Chữ số (panel 1-4)", bbox_to_anchor=(0.5, -0.02))
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig("fig13_diagnostic_9panels_mnist.png", bbox_inches="tight", dpi=120)
plt.show()

## Ảnh minh hoạ theo từng nhóm lỗi

In [ ]:
n_show = 12
groups = [TP, WR, FR, FA, TN]
rng_e = np.random.default_rng(1)
fig, axes = plt.subplots(len(groups), n_show, figsize=(1.35 * n_show, 1.7 * len(groups)))
for r, gname in enumerate(groups):
    ids = np.where(error_taxonomy == gname)[0]
    pick = rng_e.choice(ids, min(n_show, len(ids)), replace=False) if len(ids) else []
    for c in range(n_show):
        ax = axes[r, c]; ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_visible(False)
        if c < len(pick):
            i = pick[c]
            ax.imshow(X_tgt[i].reshape(28, 28), cmap="gray")
            true_lbl = str(tgt_digit[i]) + ("*" if y_spa[i] < 0 else "")
            ax.set_title(f"{true_lbl}→{pred_plot[i]}", fontsize=8, color=error_palette[gname] if gname != TN else "#555")
    axes[r, 0].set_ylabel(gname.split(":")[0] + ":" + gname.split(":")[1][:14], fontsize=7, rotation=0,
                          ha="right", va="center", labelpad=6)
fig.suptitle("Mỗi ảnh ghi: chữ số thật → nhãn dự đoán   (dấu * = chữ số bị giấu / Unknown thật)", fontsize=10)
plt.tight_layout(); plt.show()

## Bảng giải phẫu lỗi theo lớp & phân bố 5 nhóm lỗi

In [ ]:
rows_diag = []
for i, c in enumerate(class_names):
    mt = (y_spa == i)
    if not mt.any():
        continue
    wrong = raw_plot[mt & (raw_idx_full != i)]
    rows_diag.append({
        "Chữ số": c, "Số ảnh": int(mt.sum()), "Margin": round(float(margins_cls[i]), 2),
        "1_Raw_Label_Acc_%": round((raw_idx_full[mt] == i).mean() * 100, 1),
        "2_Kept_After_Filter_%": round(acc_spa[mt].mean() * 100, 1),
        "3_Final_Class_KACC_%": round((pred_idx_full[mt] == i).mean() * 100, 1),
        "Hay bị nhầm với (raw)": pd.Series(wrong).value_counts().index[0] if len(wrong) else "None",
    })
print("=== BẢNG GIẢI PHẪU LỖI TỪNG LỚP ===")
display(pd.DataFrame(rows_diag).sort_values("Số ảnh", ascending=False))

print("\n=== CHỮ SỐ BỊ GIẤU (Unknown) ĐƯỢC XỬ LÝ RA SAO ===")
for d in unknown_digits:
    mk = (tgt_digit == d)
    rej = (~acc_spa[mk]).mean() * 100
    print(f"Chữ số {d}: {mk.sum()} ảnh | bị từ chối đúng thành Unknown: {rej:.1f}% | lọt lưới: {100 - rej:.1f}%")
    absorbed = pd.Series(raw_plot[mk & acc_spa]).value_counts()
    if len(absorbed):
        print("   các lớp 'hút' nhầm (chỉ tính ảnh lọt lưới):", dict(absorbed.head(4)))

tax = pd.Series(error_taxonomy).value_counts().rename("n_cells").to_frame()
tax["%"] = (tax["n_cells"] / tax["n_cells"].sum() * 100).round(2)
print("\n=== PHÂN BỐ 5 NHÓM LỖI ===")
display(tax.sort_index())

# 10. Lưu kết quả

In [ ]:
torch.save({"model": gmm_vae.state_dict(), "label_mapping": label_mapping, "unknown_classes": unknown_digits,
            "config": CFG, "hidden_dims": hidden_dims}, "gmm_vae_mnist.pt")
np.savez("latent_mnist.npz", Z_rna=Z_rna, y_rna=y_rna, gamma_rna=gamma_rna,
         Z_spa=Z_spa, y_spa=y_spa, gamma_spa=gamma_spa, tgt_digit=tgt_digit,
         pred_raw=raw_idx_full, accept=acc_spa)
print("Đã lưu: gmm_vae_mnist.pt, latent_mnist.npz")